# MAGDA command model - LoRA fine-tune (Colab + unsloth)

Thin notebook: pulls the FROZEN dataset, LoRA fine-tunes a tiny base, exports
GGUF for the repo's llama.cpp. Logic lives in the repo, not here.

Runtime: GPU (free T4 is enough for a 0.5B LoRA). Paste cells into Colab, or
open this file directly (it's jupytext `# %%` format).

Inputs : Drive  My Drive/magda-command-model/{train,val}.chat.jsonl
         (staged from data/*.chat.jsonl via Google Drive for Desktop)
Output : command-model.gguf  saved back to the same Drive folder (auto-syncs to Mac)

## 1. Install

In [3]:
import subprocess, sys

def pip_install(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip_install("unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git")
pip_install("--no-deps", "trl", "peft", "accelerate", "bitsandbytes")

## 2. Get the data (Google Drive)
Data is staged in Drive at My Drive/magda-command-model/. We mount and read it,
and save the trained GGUF back to the same folder so it auto-syncs to the Mac
(no manual upload/download). The chat files carry the SYSTEM prompt the C++
inference path must also send (model/format.py).

In [4]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/magda-command-model"

import json, os

def load_chat(name):
    return [json.loads(l) for l in open(os.path.join(DRIVE_DIR, name), encoding="utf-8") if l.strip()]

train_rows = load_chat("train.chat.jsonl")
val_rows = load_chat("val.chat.jsonl")
print(f"train={len(train_rows)} val={len(val_rows)}")

Mounted at /content/drive
train=3027 val=400


## 3. Base model (tiny, instruct, multilingual)
Start at 0.5B; Llama-3.2-1B is the fallback. The eval back home picks the winner.

In [5]:
from unsloth import FastLanguageModel

BASE = "unsloth/Qwen2.5-0.5B-Instruct"
MAX_SEQ = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=MAX_SEQ, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=7,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-0.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.7 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


## 4. Apply the chat template

In [6]:
from datasets import Dataset

def fmt(rows):
    texts = [tokenizer.apply_chat_template(r["messages"], tokenize=False,
                                           add_generation_prompt=False) for r in rows]
    return Dataset.from_dict({"text": texts})

train_ds, val_ds = fmt(train_rows), fmt(val_rows)

## 5. Train (LoRA SFT)

In [7]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ,
        per_device_train_batch_size=16, gradient_accumulation_steps=1,
        warmup_ratio=0.05, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=20, eval_strategy="epoch", optim="adamw_8bit",
        weight_decay=0.01, lr_scheduler_type="cosine", seed=7,
        output_dir="outputs", report_to="none",
    ),
)
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3027 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


...

Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.055231,0.055251
2,0.053843,0.052354
3,0.050121,0.051695


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-570/tokenizer_config.json.


TrainOutput(global_step=570, training_loss=0.1970579250862724, metrics={'train_runtime': 1663.4002, 'train_samples_per_second': 5.459, 'train_steps_per_second': 0.343, 'total_flos': 3672134351504640.0, 'train_loss': 0.1970579250862724, 'epoch': 3.0})

## 6. Sanity check

In [8]:
FastLanguageModel.for_inference(model)
SYSTEM = train_rows[0]["messages"][0]["content"]
for req in ["create a bass track with serum and ott", "mute Drums", "把贝斯轨道设为蓝色"]:
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": req}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=128, temperature=0.0)
    print(req, "->", tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ValueError: `temperature` (=0.0) has to be a strictly positive float, otherwise your next token scores will be invalid. If you're looking for greedy decoding strategies, set `do_sample=False`.

## 7. Export GGUF for llama.cpp -> Drive (auto-syncs to Mac)
Then back home:
  python -m eval.run --model "~/.../My Drive/magda-command-model/command-model...gguf"

In [ ]:
import glob, shutil

model.save_pretrained_gguf("command-model", tokenizer, quantization_method="q4_k_m")

for f in glob.glob("command-model/*.gguf") + glob.glob("*.gguf"):
    dst = shutil.copy(f, DRIVE_DIR)
    print("synced to Drive ->", dst)